In [2]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

First, you need to upload the CSV file from your local machine to the Colab environment. You can use the `files.upload()` function from `google.colab` for this.

In [3]:
from google.colab import files
import io

uploaded = files.upload()

# Assuming you upload a file named 'your_file_name.csv'
# You will need to replace 'your_file_name.csv' with the actual name of your uploaded file.

Saving Features_For_Traditional_ML_Techniques.csv to Features_For_Traditional_ML_Techniques (1).csv


Once the file is uploaded, you can read it into a pandas DataFrame. Make sure to replace `'your_file_name.csv'` with the exact name of the file you uploaded.

In [4]:
# Get the name of the uploaded file(s)
# This assumes only one file is uploaded. If multiple, you'll need to choose the correct one.
for fn in uploaded.keys():
  print(f'User uploaded file "{fn}"')
  truthseeker_df = pd.read_csv(io.StringIO(uploaded[fn].decode('utf-8')))

# Display the first few rows of the DataFrame, shape and headers
display(truthseeker_df.head())
print(truthseeker_df.shape)

# print(truthseeker_df['tweet'][0])
print(truthseeker_df.info()) # print df info

User uploaded file "Features_For_Traditional_ML_Techniques (1).csv"


,Unnamed: 0,majority_target,statement,BinaryNumTarget,tweet,followers_count,friends_count,favourites_count,statuses_count,listed_count,...,determiners,conjunctions,dots,exclamation,questions,ampersand,capitals,digits,long_word_freq,short_word_freq
0,0,True,End of eviction moratorium means millions of A...,1.0,@POTUS Biden Blunders - 6 Month Update\n\nInfl...,4262.0,3619.0,34945.0,16423.0,44.0,...,0,0,5,0,1,0,33,3,5,19
1,1,True,End of eviction moratorium means millions of A...,1.0,@S0SickRick @Stairmaster_ @6d6f636869 Not as m...,1393.0,1621.0,31436.0,37184.0,64.0,...,0,2,1,0,0,0,14,0,2,34
2,2,True,End of eviction moratorium means millions of A...,1.0,THE SUPREME COURT is siding with super rich pr...,9.0,84.0,219.0,1184.0,0.0,...,0,1,0,0,0,0,3,0,4,10
3,3,True,End of eviction moratorium means millions of A...,1.0,@POTUS Biden Blunders\n\nBroken campaign promi...,4262.0,3619.0,34945.0,16423.0,44.0,...,0,1,3,0,0,1,6,8,1,30
4,4,True,End of eviction moratorium means millions of A...,1.0,@OhComfy I agree. The confluence of events rig...,70.0,166.0,15282.0,2194.0,0.0,...,0,1,3,0,1,0,11,3,2,19


(134198, 64)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 134198 entries, 0 to 134197
Data columns (total 64 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   Unnamed: 0              134198 non-null  int64  
 1   majority_target         134198 non-null  bool   
 2   statement               134198 non-null  object 
 3   BinaryNumTarget         134198 non-null  float64
 4   tweet                   134198 non-null  object 
 5   followers_count         134198 non-null  float64
 6   friends_count           134198 non-null  float64
 7   favourites_count        134198 non-null  float64
 8   statuses_count          134198 non-null  float64
 9   listed_count            134198 non-null  float64
 10  following               134198 non-null  float64
 11  embeddings              134198 non-null  object 
 12  BotScore                134198 non-null  float64
 13  BotScoreBinary          134198 non-null  float64
 14  cred   

### 2. Define Features (X) and Target (y)
We will use `majority_target` as our target variable (y) and the rest of the relevant columns as features (X). We'll exclude text-based columns (`statement`, `tweet`) for this initial SVC model, as they require specialized NLP techniques.

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Define target (y) and features (X)
y = truthseeker_df['majority_target']

# Drop target, uneeded, and text columns for now
X = truthseeker_df.drop(columns=['majority_target', 'statement', 'tweet', 'embeddings', 'BinaryNumTarget', 'Unnamed: 0'])

# Handle potential NaN values if any (e.g., fill with mean or median)
X = X.fillna(X.mean(numeric_only=True))

print(f"Shape of X: {X.shape}")
print(f"Shape of y: {y.shape}")

Shape of X: (134198, 58)
Shape of y: (134198,)


### 3. Split Data into Training and Testing Sets
We'll split the data to evaluate the model's performance on unseen data.

In [6]:
sample_fraction = 0.01 # Adjust this value to change the sample size (e.g., 0.1 for 10%)

# Sample the data before splitting
X_sample, y_sample = X.sample(frac=sample_fraction, random_state=42), y.sample(frac=sample_fraction, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X_sample, y_sample, test_size=0.2, random_state=42, stratify=y_sample)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (1073, 58)
X_test shape: (269, 58)
y_train shape: (1073,)
y_test shape: (269,)


### 4. Scale Features
SVC models are sensitive to the scale of input features, so we'll use `StandardScaler` to normalize them.

In [7]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled successfully.")

Features scaled successfully.


### 5. Train and Evaluate SVC Model
Now, we'll initialize and train the Support Vector Classifier and then evaluate its performance.

In [8]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

# Initialize the SVC model
# Using 'linear' kernel for potentially faster training on larger datasets, but 'rbf' is also common.
# You might need to experiment with different kernels and parameters.
svc_model = SVC(kernel='linear', random_state=42, C=1.0)

# Train the model
print("Training SVC model...")
svc_model.fit(X_train_scaled, y_train)
print("SVC model training complete.")

# Make predictions on the test set
y_pred = svc_model.predict(X_test_scaled)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"\nModel Accuracy: {accuracy:.4f}")
print("\nClassification Report:\n", report)

Training SVC model...
SVC model training complete.

Model Accuracy: 0.5576

Classification Report:
               precision    recall  f1-score   support

       False       0.56      0.55      0.55       135
        True       0.55      0.57      0.56       134

    accuracy                           0.56       269
   macro avg       0.56      0.56      0.56       269
weighted avg       0.56      0.56      0.56       269

